# HW 5: Extra Credit

In this assignment, we will train a transformer model to generate images autoregressively at the pixel level, which means the model will predict one pixel at a time in a preconfigured order ([raster scan order](https://en.wikipedia.org/wiki/Raster_scan#:~:text=Scan%20lines,-In%20a%20raster&text=This%20ordering%20of%20pixels%20by,continuously%20over%20the%20scan%20line)). Like previous assignments, we will train our model on the MNIST dataset.


The homework is broken up as follows:

  1. [Question 1: Tokenization (1.5 points)](#1)  
  1. [Question 2: Transformer Implementation (1.5 points)](#2)  
  1. [Question 3: Transformer Training (2.5 points)](#3)  
  1. [Question 4: Transformer Inference (2.5 points)](#4)  
  1. [Question 5: Image Completions (2 points)](#5)  

## How to Submit This Homework

This homework has been provided to you as an `.ipynb` file. I expect you to complete the notebook and submit the following on Canvas:

1. The completed notebook itself with your answers filled in (including any source code used to answer the question)
2. A `.pdf` file of the notebook that shows all your answers. This will make grading easier and I will appreciate that.

If you submit only one of these, you will lose points. No late homework will be graded. No submissions via email or other media will be graded.

__Finally, each question specifies exactly what should be provided in your answer -- for example, if no code is requested, do not provide it. Your submitted notebook (and PDF) should not contain any additional content or cells beyond what is requested.__

## Setup

If you do not have access to a CUDA-compatible NVIDIA GPU, it is recommended that you run this notebook in [Google Colab](https://colab.research.google.com/). There, you will have the option to enable GPU acceleration with `Runtime` >> `Change runtime type` >> `Hardware accelerator` >> `GPU` >> `Save`. Note that you may have to re-connect or restart the notebook after changing runtime types.

In [ ]:
# helper code from the course repository
!git clone https://github.com/interactiveaudiolab/course-deep-learning.git
# install common pacakges used for deep learning
!cd course-deep-learning/ && pip install -r requirements.txt

In [ ]:
%matplotlib inline
%cd course-deep-learning/

import datetime
import math
import torch
import matplotlib.pyplot as plt
import numpy as np
import IPython.display as ipd
from matplotlib.animation import FuncAnimation
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
from torchsummary import summary
from tqdm import tqdm

from utils.adversarial_examples import *

----
## <a name="1">Question 1: Tokenization (1.5 points total)</a>

Recall that autoregressive language models typically predict a sequence of discrete (integer) tokens from a fixed vocabulary. At first glance, image data lacks this discrete sequential structure and therefore seems like a poor candidate for language modeling.

However, there are in fact many ways to represent an image as a sequence of discrete tokens. [State-of-the art approaches](https://arxiv.org/abs/2404.02905) often leverage large neural network-based autoencoders, but we'll keep things as simple as possible: first, we will "flatten" an image into a one-dimensional vector of pixels, and then we will treat each pixel as a token by rounding its continuous intensity value to one of a few discrete values. To do this, we can divide the observed range of intensities into bins, and represent a pixel's intensity through its bin index. This is the approach we'll be taking.

We will implement a `Tokenizer` object that converts image data into tokens. This object-oriented approach mimics the tokenizer abstraction found in most language modeling repositories, across modalities from text to audio to images.


### 1.1: Implementing Tokenization

**YOUR TASK (1 point)**: Fill in the commented parts of the code below to tokenize 2D images with continuous values into 1D sequences of discrete tokens. Your code should divide the observed range of intensity values into equal-width bins that uniformly cover the observed range, and you should set your bins using the MNIST _validation_ dataset.

### 1.2: Tokenization Results

**YOUR TASK (0.5 points)**: After tokenizing the MNIST _training_ dataset, plot a histogram of the most common tokens by iterating through every image and counting the number of times each token appears. Matplotlib may cause your notebook kernel error to out when trying to create a very large histogram, so use the first 1,000,000 tokens.

In [ ]:
class Tokenizer(torch.nn.Module):

  def __init__(
    self,
    n_channels: int = 1,  # MNIST digits are grayscale so there is only 1 channel
    image_size: int = 28, # MNIST digits are 28 by 28, so this is really image_height and image_width. 
    vocab_size: int = 16  # This is the number of quantization bins we'll be using when we tokenize
  ):
    super().__init__()

    self.n_channels = n_channels
    self.image_size = image_size
    self.vocab_size = vocab_size 

    self.initialized = False

    self.bins = None

  def initialize(self, images: list):

    ########################################
    # YOUR CODE HERE: given a list of image tensors,
    # divide the image intensity range
    # into `self.n_vocab_size` equally-
    # spaced intensity bins and store
    # these in `self.bins`
    ########################################

    ...

    self.bins = ???
    self.initialized = True

  def forward(self, x: torch.Tensor):

    if not self.initialized:
      raise RuntimeError(
        f"Tokenizer not initialized! Must call `.initialize()` "
        f"on a list of image tensors to set bins first"
      )

    assert x.ndim == 4  # (n_batch, n_channels, image_size, image_size)
    assert x.shape[1] == self.n_channels
    assert x.shape[2] == x.shape[3] == self.image_size

    # flatten image into pixel sequence
    x_flat = x.reshape(x.shape[0], -1)  # (n_batch, n_channels * image_size * image_size)

    ########################################
    # YOUR CODE HERE: apply tokenization by replacing
    # each pixel value in `x_flat` with the
    # integer index of the nearest intensity
    # bin in `self.bins`. The resulting
    # tensor should be of dtype `torch.long`
    # and have the same shape as `x_flat`
    ########################################

    ...
    x_tokenized = ???

    return x_tokenized  # (n_batch, n_channels * image_size * image_size)


torch.manual_seed(0)  # fix random seed

# select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# download MNIST data
mnist_train, mnist_test = load_mnist()

tokenizer = Tokenizer(n_channels=1, image_size=28, vocab_size=16)

# loop over a few batches to estimate bins
images = []
for batch in mnist_test:
  x, _ = batch
  images += [x]

images = torch.cat(images, dim=0)
images = [images[i:i+1] for i in range(images.shape[0])]

tokenizer.initialize(images)

print("Bins", tokenizer.bins)

# plot an example image before and after tokenization,
# accounting for the fact that our tokenizer "flattens"
# images
plt.imshow(x[0][0])
plt.show()

x_tok = tokenizer(x)
x_tok = x_tok.reshape(-1, 1, 28, 28)
plt.imshow(x_tok[0][0])
plt.show()

print("Example tokenized image tensor")
print(x_tok[0])


########################################
# YOUR CODE HERE: Write the code to make the 
# histogram of token frequencies
########################################

<center>
<div style="color:red;background-color:#e5e5e5">(YOUR HISTOGRAM HERE)</div>
</center>

----
## <a name="2">Question 2: Transformer Implementation (1.5 points total)</a>

Next, we’ll build our transformer architecture. We will implement a _decoder-only_ (_GPT-style_) transformer model that takes a sequence of image tokens as input and outputs a probability distribution for what the next token in the sequence will be.

Here are some useful general resources when working with transformers; note that these articles describe _encoder-decoder_ transformers, which differ slightly from the model we will be implementing:
* Jay Alammar’s [Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)  
* [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/)  
* [The Original Transformer Paper](https://arxiv.org/pdf/1706.03762)  

This article focuses on the decoder-only transformer architecture that we'll be using:
* [Decoder-Only Transformers](https://cameronrwolfe.substack.com/p/decoder-only-transformers-the-workhorse)

We've defined most of the architecture for you below. However, we've omitted one key part: _the self-attention mechanism_. The articles linked above discuss how this mechanism works in detail.


**YOUR TASK (1.5 points)**: Fill in the commented part of the `MHSA` class code below according to the instructions. You can test your implementation by running the cell below.

In [ ]:
class MLP(torch.nn.Module):
  """
  The MLP block within a transformer layer. Can be thought to "mix" the outputs
  of multiple attention heads at each sequence position, without mixing 
  information from across separate sequence positions.
  """
    
  def __init__(
    self,
    model_dim: int,          # the model's internal "dimension" or "hidden size"
    mult: int = 4,           # the number of times bigger the MLP's hidden dimension will be than "model_dim"
    p_dropout: float = 0.1,  # probability of random dropout
  ):

    super().__init__()

    self.mlp = torch.nn.Sequential(
      torch.nn.Linear(model_dim, model_dim * mult),
      torch.nn.Dropout(p_dropout),
      torch.nn.GELU(),
      torch.nn.Linear(model_dim * mult, model_dim),
    )

  def forward(self, x: torch.Tensor):

    assert x.ndim == 3  # (n_batch, seq_len, model_dim)

    return self.mlp(x)  # (n_batch, seq_len, model_dim)


class PositionalEncoding(torch.nn.Module):
  """
  Sinusoidal positional encoding, which is added to input embeddings before
  they are processed with transformer layers.
  """

  def __init__(
    self,
    model_dim: int,  # the model's internal "dimension" or "hidden size"
    max_length: int = 1024,  # the maximum allowed sequence length for our transformer
  ):
    super().__init__()

    self.model_dim = model_dim
    self.max_length = max_length

    pe = torch.zeros(max_length, model_dim)
    position = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)

    scale = torch.exp(
      torch.arange(
          0, model_dim, 2, dtype=torch.float
      ) * (-math.log(10000.0) / model_dim)
    )
    pe[:, 0::2] = torch.sin(position * scale)
    pe[:, 1::2] = torch.cos(position * scale)

    pe = pe.unsqueeze(0)  # (1, max_length, model_dim)
    self.register_buffer('pe', pe)

  def forward(self, x: torch.Tensor):

    assert x.ndim == 3  # (n_batch, seq_len, model_dim)
    assert x.shape[1] <= self.max_length
    assert x.shape[2] == self.model_dim

    seq_len = x.size(1)
    x = x + self.pe[:, :seq_len]

    return x  # (n_batch, seq_len, model_dim)


class MHSA(torch.nn.Module):
  """
  This class implements multi-headed self-attention (MHSA), and is where you 
  will be making changes to the code!

  Multi-headed self-attention divides embedding vectors in a given sequence 
  into groups along the "channel" or "embedding" dimension; each "head" then
  processes a single group, with all heads operating independently in parallel.
  The outputs from all heads are then concatenated and mixed together using a
  subsequent MLP block.
  """
  def __init__(
    self,
    model_dim: int,  # the model's internal "dimension" or "hidden size"
    n_heads: int,  # the number of attention heads to apply in parallel
    p_dropout: float = 0.1,  # probability of random dropout
  ):
    super().__init__()

    assert model_dim % n_heads == 0
    head_dim = model_dim // n_heads

    self.model_dim = model_dim
    self.n_heads = n_heads
    self.head_dim = head_dim

    self.q_proj = torch.nn.Linear(model_dim, model_dim)
    self.k_proj = torch.nn.Linear(model_dim, model_dim)
    self.v_proj = torch.nn.Linear(model_dim, model_dim)

    self.dropout = torch.nn.Dropout(p_dropout)

    self.out_proj = torch.nn.Linear(model_dim, model_dim)

  def forward(self, x: torch.Tensor, attn_mask: torch.Tensor = None):

    assert x.ndim == 3  # (n_batch, seq_len, model_dim)
    n_batch, seq_len, model_dim = x.shape

    q = self.q_proj(x)
    k = self.k_proj(x)
    v = self.v_proj(x)

    q = q.view(n_batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
    k = k.view(n_batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
    v = v.view(n_batch, seq_len, self.n_heads, self.head_dim).transpose(1, 2)

    if attn_mask is None:
      attn_mask = torch.ones(1, 1, seq_len, seq_len, device=x.device, dtype=torch.bool)
    else:
      assert attn_mask.ndim == 4
      attn_mask = attn_mask.bool()

    ################################################################
    # YOUR CODE HERE: implement the core attention operation as follows:
    # 1. Convert our attention mask into an
    #    attention bias by replacing all `True` entries with 0 and
    #    all `False` entries with -1e9.
    # 2. Compute attention logits (unnormalized weights) by
    #    performing a matrix multiplication between the query and
    #    key tensors
    # 3. Scale attention logits by dividing by the square root of
    #    `self.head_dim`
    # 4. Add our attention bias to our attention logits, effectively
    #    lowering the weights of masked positions we do not want to
    #    attend to
    # 5. Apply softmax along the final dimension of these biased
    #    attention logits, giving us normalized attention weights
    # 6. Apply `self.dropout` to these attention weights
    # 7. Finally, perform a weighted sum of values in our value 
    #    tensor using our attention weights. If you want, this can 
    #    be implemented using matrix multiplication between our
    #    attention weights and the value tensor
    ################################################################

    # 1. convert mask to additive bias
    attn_bias = ???

    assert attn_bias.shape == attn_mask.shape
    assert torch.all(attn_bias[~attn_mask] == -1e9)

    # 2. compute attention logits via matmul
    # 3. scale attention logits
    attn_logits = ???

    assert attn_logits.shape == (n_batch, self.n_heads, seq_len, seq_len)

    # 4. add attention bias to scaled logits
    attn_logits = ???

    assert attn_logits.shape == (n_batch, self.n_heads, seq_len, seq_len)

    # 5. apply softmax along final dimension to normalize weights
    attn_weights = ???

    assert attn_weights.shape == (n_batch, self.n_heads, seq_len, seq_len)

    # 6. apply dropout to attention weights
    attn_weights = ???

    assert attn_weights.shape == (n_batch, self.n_heads, seq_len, seq_len)

    # 7. weighted sum of values using attention weights
    attn_out = ???

    assert attn_out.shape == (n_batch, self.n_heads, seq_len, self.head_dim)

    # ... and we're done!

    # combine heads
    attn_out = attn_out.transpose(1, 2).contiguous()
    attn_out = attn_out.view(n_batch, seq_len, model_dim)  # (n_batch, seq_len, model_dim)

    output = self.out_proj(attn_out)  # (n_batch, seq_len, model_dim)

    return output


def get_causal_mask(x: torch.Tensor):
  """
  A utility function for computing a causal attention mask given an input
  sequence. This mask will have shape (n_batch, 1, seq_len, seq_len), where
  seq_len is the sequence length of the input. We can think of this mask as a 
  square grid, where an entry at index [i, j] is True if we want to allow
  sequence position i to attend to sequence position j, and False otherwise.

  To prevent out transformer attention layers from "looking forward" in time,
  we will construct a mask such that the entry at index [i, j] will be False
  when i < j, and True otherwise.
  """
  assert x.ndim == 3  # (n_batch, seq_len, model_dim)
  n_batch, seq_len, model_dim = x.shape

  mask = torch.tril(
    torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device)
  )
  mask = mask.unsqueeze(0).unsqueeze(0).expand(n_batch, 1, seq_len, seq_len)
  return mask  # (n_batch, 1, seq_len, seq_len)


class TransformerLayer(torch.nn.Module):
  """
  We combine the blocks defined above into a transformer layer, which applies
  normalization, multi-headed self-attention, normalization, and then an MLP
  block to inputs.

  The transformer layer also includes residual (skip) connections between these
  operations, allowing for improved information flow.

  Given a sequence input of shape (n_batch, seq_len, model_dim) and an optional
  attention mask, the transformer layer produces an output sequence of the
  same shape -- hopefully, one that has been processed in a way to reveal
  useful contextual information.
  """
    
  def __init__(
      self,
      model_dim: int,  # the model's internal "dimension" or "hidden size"
      n_heads: int,    # the number of attention heads to apply in parallel
      p_dropout: int,  # probability of random dropout
      mult: int,       # the number of times bigger the MLP's hidden dimension will be than "model_dim"
  ):
    super().__init__()

    self.mlp = MLP(
      model_dim=model_dim,
      p_dropout=p_dropout,
      mult=mult,
    )

    self.mhsa = MHSA(
      model_dim=model_dim,
      n_heads=n_heads,
      p_dropout=p_dropout,
    )

    self.norm1 = torch.nn.LayerNorm(model_dim)
    self.norm2 = torch.nn.LayerNorm(model_dim)

  def forward(self, x: torch.Tensor, attn_mask: torch.Tensor = None):

    assert x.ndim == 3  # (n_batch, seq_len, model_dim)

    x = x + self.mhsa(self.norm1(x), attn_mask)
    x = x + self.mlp(self.norm2(x))

    return x


class TransformerDecoder(torch.nn.Module):
  """
  This is our final transformer model, consisting of a stack of transformer 
  layers.

  Before we apply our transformer layers to an input sequence one-at-a-time,
  we will first convert out sequence from discrete tokens to continuous 
  embedding vectors of size "model_dim".

  After applying our transformer layers, we will pass our output sequence
  through a final linear (projection) layer to convert each vector in the
  sequence into an unnormalized probability distribution over tokens in the
  vocabulary.
  """
  def __init__(
    self,
    vocab_size: int,  # the number tokens (quantized pixel values) in our tokenized vocabulary
    n_layers: int,    # the number of transformer layers
    model_dim: int,   # the model's internal "dimension" or "hidden size"
    max_length: int,  # the maximum allowed sequence length for our transformer
    n_heads: int,     # the number of attention heads to apply in parallel
    p_dropout: int,   # probability of random dropout
    mult: int,        # the number of times bigger the MLP's hidden dimension will be than "model_dim"
  ):
    super().__init__()

    self.emb = torch.nn.Embedding(vocab_size, model_dim)

    self.pos_enc = PositionalEncoding(
        model_dim=model_dim,
        max_length=max_length
    )

    self.layers = torch.nn.ModuleList(
        [
            TransformerLayer(
                model_dim=model_dim,
                n_heads=n_heads,
                p_dropout=p_dropout,
                mult=mult,
            )
            for _ in range(n_layers)
        ]
    )

    self.out_proj = torch.nn.Linear(model_dim, vocab_size)

  def forward(self, x: torch.Tensor):
    assert x.ndim == 2  # (n_batch, seq_len)

    x = self.emb(x)  # (n_batch, seq_len, model_dim)

    x = self.pos_enc(x)

    attn_mask = get_causal_mask(x)

    for layer in self.layers:
        x = layer(x, attn_mask)

    out = self.out_proj(x)

    return out

You can test your code below:

In [ ]:
mhsa = MHSA(
  model_dim=4,
  n_heads=2,
  p_dropout=0.0,
)

for p in mhsa.parameters():
    torch.nn.init.constant_(p, 0.1)

x = torch.tensor(
  [[[0.1, 0.2, 0.3, 0.4],],[[0.5, 0.6, 0.7, 0.8],],]
) # (n_batch==2, seq_len==1, model_dim==4)

mask = torch.tensor(
  [[[[False],],[[True],],],[[[False],],[[True],],],], dtype=torch.bool
)  # (n_batch==2, n_heads==2, seq_len==1, seq_len==1)

with torch.no_grad():
  out = mhsa(x, mask)

assert out.shape == x.shape
assert torch.allclose(
    out, torch.tensor(
      [[[0.1800, 0.1800, 0.1800, 0.1800],],[[0.2440, 0.2440, 0.2440, 0.2440],],]
    ),
    atol=1e-4
)



----
## <a name="3">Question 3: Transformer Training (2.5 points total)</a>

Now, it’s time to train our model. In the training loop, we will receive a batch of images from the MNIST dataset, which we will tokenize (i.e. flatten and convert to integer tokens) and then further process to get both our inputs and our targets for the model.

Given a token sequence, a transformer can be trained on all possible "next token prediction" tasks in the sequence _in parallel_ -- i.e., predicting the second token given the first, predicting the third token given the first and second, predicting the fourth token given the first, second, and third, etc. This is possible through _causal masking_, which is implemented in our code above and discussed in the linked articles.

To take advantage of this, we simply need to format our data as an _input sequence_ that we will pass to the transformer to represent the "tokens up through the current position", and a _target sequence_ for loss computation that will represent the "next token" to predict at each sequence position. Because there is no "previous token" at the first sequence position, we will prepend a special `<START>` token to our input. Similarly, because we do not predict any further tokens after the final token in our sequence, we will trim the final sequence token from our input.

Note that our tokenizer does not produce sequences with a `<START>` token, as it only converts pixels to corresponding token indices in the range `[0, VOCAB_SIZE - 1]`. Therefore, we will use the token index `VOCAB_SIZE` as our `<START>` token, and correspondingly use `vocab_size=VOCAB_SIZE+1` when initializing our transformer model to account for this extra possible token in our embedding and final projection layers.



### 3.1: Formatting Inputs \& Targets

**YOUR TASK (1 point)**: Fill in the required parts of the code below to format the input and target sequences as described above.

### 3.2: Teacher Forcing

**YOUR TASK (1 point)**: Explain what teacher forcing is. Are the (input, target) pairs generated here a form of teacher-forcing?

<center>
<div style="color:red;background-color:#e5e5e5">(YOUR ANSWER HERE)</div>
</center>

### 3.3: Training

**YOUR TASK (0.5 point)**: Train your model for at least 3 epochs using the cell below, and use the next cell to generate plots of your training losses and accuracies. On a Google Colab T4 runtime, this should take about 20 minutes.

#### Training Configuration
Use these training hyperparameters. __Do not edit this cell.__

In [ ]:
VOCAB_SIZE = 16         # This is the number of quantization 'tokens' we'll use
MODEL_DIM = 128         # the model's internal "dimension" or "hidden size"
N_LAYERS = 4            # The number of transformer blocks 
N_HEADS = 4             # The number of attention heads
MULT = 4                # This is used in the MLP module to govern the number of nodes in its hidden layer 
MAX_LENGTH = 28 * 28    # Since MNIST digits are 28 by 28, and we flatten them, this is our max sequence length
P_DROPOUT = 0.1         # Probability of dropout 

torch.manual_seed(0)  # fix random seed

# select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# download MNIST data
mnist_train, mnist_test = load_mnist()


def count_parameters(model: torch.nn.Module):
    return sum(p.numel() for p in model.parameters())

#### Training Loop

__Modify and run this cell.__

In [ ]:

# initialize transformer decoder
tf = TransformerDecoder(
  vocab_size=VOCAB_SIZE + 1,  # Account for "start" token
  model_dim=MODEL_DIM,
  n_layers=N_LAYERS,
  mult=MULT,
  max_length=MAX_LENGTH,
  n_heads=N_HEADS,
  p_dropout=P_DROPOUT,
)
tf.to(device)

# initialize image tokenizer
tokenizer = Tokenizer(
    n_channels=1,
    image_size=28,
    vocab_size=VOCAB_SIZE,
)

# loop over a few batches to estimate bins
it = iter(mnist_test)
images = []
for i in range(10):
  x, _ = next(it)
  images += [x]

images = torch.cat(images, dim=0)
images = [images[i:i+1] for i in range(images.shape[0])]

tokenizer.initialize(images)

print("Transformer parameters:", count_parameters(tf))

opt = torch.optim.AdamW(tf.parameters(), lr=3e-4)

losses = []
accuracies = []

for epoch in range(10):

  # Training epoch
  tf.train()

  pbar = tqdm(mnist_train)
  for batch in pbar:

    opt.zero_grad()

    x, _ = batch

    ########################################
    # YOUR CODE HERE: format inputs and targets as
    # follows:
    #
    # 1. Tokenize your sequence
    # 2. Prepend the start token VOCAB_SIZE
    #    to the beginning of your input
    #    sequence
    # 3. Trim the final token from your
    #    input sequence
    # 4. Use your unmodified token sequence
    #    as your training target
    #
    # After this, your model's input should
    # be in the variable `x`, and your
    # target should be in the variable `y`
    ########################################

    # 1. Tokenize sequence
    ...

    # 2. Prepend start token
    ...

    # 3. Trim final token
    ...

    # 4. Use unmodified token sequence as target
    ...

    # ... and we're done!
    assert x.ndim == 2
    assert x.shape[-1] == MAX_LENGTH
    assert y.ndim == 2
    assert y.shape[-1] == MAX_LENGTH

    pred = tf(x.to(device))  # (n_batch, seq_len, vocab_size)

    l = torch.nn.functional.cross_entropy(pred.transpose(1, 2), y.to(device)).mean()

    with torch.no_grad():
      acc = (pred.argmax(dim=-1) == y.to(device)).float().mean()

    l.backward()
    opt.step()

    pbar.set_description(f"Loss: {l.item() :0.3f}, Accuracy: {acc.item() :0.3f}")

    losses += [l.item()]
    accuracies += [acc.item()]

In [ ]:
plt.plot(losses)
plt.title("Training Loss")
plt.show()

plt.plot(accuracies)
plt.title("Training Accuracy")
plt.show()

----
## <a name="4">Question 4: Transformer Inference (2.5 points total)</a>

Now that we've trained our model for next-token prediction, let's see if we can use it to generate MNIST images one token (pixel) at a time.

### 4.1: ArgMax Inference

**YOUR TASK (1 point)**: Fill in the commented portion of the `inference_argmax` code below to complete the autoregressive generation loop, and generate __two__ example images by running the cell.

In [ ]:
@torch.no_grad()
def inference_argmax(
  m: TransformerDecoder,
  x: torch.Tensor = None,
  max_length: int = 784,
):
  m.eval()

  # start with a partial sequence; if no partial sequence is
  # given, start with only the "start" token
  if x is None:
    x = torch.full((1, 1), VOCAB_SIZE)
  else:
    assert x.ndim == 2
    assert x.shape[0] == 1

  seq_len = x.shape[1]

  remaining = max_length - seq_len + 1  # account for "start" token

  x = x.to(device)

  for i in tqdm(range(remaining)):

    out = m(x)  # (n_batch, partial_seq_len, vocab_size)

    ########################################
    # YOUR CODE HERE: complete the argmax inference
    # loop by doing the following
    #
    # 1. Select the logits for the last
    #    sequence step in our transformer's
    #    output
    # 2. Take the argmax over the vocabulary
    #    dimension of these logits to sample
    #    an integer token
    # 3. Append this sampled token to our
    #    input x
    ########################################

    # 1. Select logits for last step
    ...

    # 2. Take argmax to sample token
    ...

    # 3. Append sampled token to input
    ...

  return x[:, 1:]


out = inference_argmax(tf)

plt.imshow(out.reshape(28, 28).cpu().detach())
plt.show()

<center>
<div style="color:red;background-color:#e5e5e5">(YOUR TWO IMAGES HERE)</div>
</center>

### 4.2: Multinomial Inference

**YOUR TASK (1 point)**: Complete the `inference_multinomial` code below to apply random multinomial sampling rather than argmax sampling of each token in the autoregressive generation loop, using `torch.multinomial`. Generate __two__ example images by running the cell.


In [ ]:
@torch.no_grad()
def inference_multinomial(
  m: TransformerDecoder,
  x: torch.Tensor = None,
  max_length: int = 784,
):
  m.eval()

  # start with a partial sequence; if no partial sequence is
  # given, start with only the "start" token
  if x is None:
    x = torch.full((1, 1), VOCAB_SIZE)
  else:
    assert x.ndim == 2
    assert x.shape[0] == 1

  seq_len = x.shape[1]

  remaining = max_length - seq_len + 1  # account for "start" token

  x = x.to(device)

  for i in tqdm(range(remaining)):

    out = m(x)  # (n_batch, partial_seq_len, vocab_size)

    ########################################
    # YOUR CODE HERE: complete the multinomial
    # inference loop by doing the following
    #
    # 1. Select the logits for the last
    #    sequence step in our transformer's
    #    output
    # 2. Apply softmax along the vocabulary
    #    dimension to convert logits to
    #    token probabilities
    # 3. Sample an integer index (token)
    #    from these probabilities using
    #    `torch.multinomial`
    # 4. Append this sampled token to our
    #    input x
    ########################################

    # 1. Select logits for last step
    ...

    # 2. Apply softmax along vocabulary dimension
    ...

    # 3. Use `torch.multinomial` to sample token
    ...

    # 3. Append sampled token to input
    ...

  return x[:, 1:]


out = inference_multinomial(tf)

plt.imshow(out.reshape(28, 28).cpu().detach())
plt.show()

<center>
<div style="color:red;background-color:#e5e5e5">(YOUR TWO IMAGES HERE)</div>
</center>

### 4.3: Argmax vs. Multinomial

**YOUR TASK (0.5 points)**: Explain whether you see any difference between the quality of outputs generated with argmax and multinomial sampling; if so, what might account for this difference?

<center>
<div style="color:red;background-color:#e5e5e5">(YOUR ANSWER HERE)</div>
</center>

----
## <a name="5">Question 5: Image Completion (2 points total)</a>

Finally, we'll explore how our "raster-order" autoregressive generation of pixels can be used to complete partial images.

### 5.1: Completing an 8

**YOUR TASK (1 point)**: Find an image of the digit `8` from the MNIST _validation_ set and trim rows from the bottom until only part of the "top circle" remains. Pass a corresponding token sequence to one of your two `inference` functions from Question 4 to generate the remainder of the image, and __provide one generation that looks like an 8, and one that looks like a 0__. Make sure you also plot the original 8, as well as the partial (trimmed) 8 on which completions are performed.



In [ ]:
...

<center>
<div style="color:red;background-color:#e5e5e5">(YOUR ORIGINAL "8" IMAGE HERE)</div>
</center>

<center>
<div style="color:red;background-color:#e5e5e5">(YOUR TRIMMED "8" IMAGE HERE)</div>
</center>

<center>
<div style="color:red;background-color:#e5e5e5">(YOUR "8" COMPLETED GENERATION HERE)</div>
</center>

<center>
<div style="color:red;background-color:#e5e5e5">(YOUR "0" COMPLETED GENERATION HERE)</div>
</center>

### 5.2: Generation Order

**YOUR TASK (1 point)**: Can our current trained transformer generate images from bottom-to-top? If so, how? If not, what could we modify to generate in this order?

<center>
<div style="color:red;background-color:#e5e5e5">(YOUR ANSWER HERE)</div>
</center>